<img src="assets/OFE-color-horizontal.png" width="300">

# Running a relative binding free energy campaign with OpenFE

*OpenFE: CCPBiosim 2026 Training Week — 2nd October 2026.*

In this notebook we demonstrate how you can use [OpenFE](https://openfree.energy/) to plan,
set up, run, and analyse a relative binding free energy (RBFE) campaign for a series of congeneric
ligands. First with the OpenFE command line interface (CLI) using OpenFE's recommended defaults,
and then using the Python API to build a more customised campaign.

By the end of this tutorial you will be able to:

- Explain the core concepts of a hybrid topology RBFE calculation.
- Plan an entire RBFE campaign using OpenFE's CLI commands.
- Use the OpenFE Python API to build a custom campaign: choosing your own atom mapper, network
  topology, protocol settings, and partial charge method.
- Know where to look to run and analyse the resulting simulations.

For more information on how to use OpenFE, we refer you to:

- The [OpenFE documentation](https://docs.openfree.energy/en/stable/). We particularly recommend the OpenFE [tutorials](https://docs.openfree.energy/en/stable/tutorials/index.html), [cookbooks](https://docs.openfree.energy/en/stable/cookbook/index.html), and [user guide](https://docs.openfree.energy/en/stable/guide/index.html).
- The [OpenFE discussions board](https://github.com/orgs/OpenFreeEnergy/discussions).
- The [OpenFE source code](https://github.com/OpenFreeEnergy/openfe).

We also recommend the following literature:
- Mey et al., ["Best Practices for Alchemical Free Energy Calculations"](https://livecomsjournal.org/index.php/livecoms/article/view/v2i1e18378), Living J. Comp. Mol. Sci., 2020
- Baumann et al., [Large-Scale Collaborative Assessment of Binding Free Energy Calculations for Drug Discovery Using OpenFE](https://pubs.acs.org/jcisd8/article-abstract/66/11/6429/5185029/Large-Scale-Collaborative-Assessment-of-Binding?redirectedFrom=fulltext), J. Chem. Inf. Model., 2026

## Overview

| Section | What we'll cover |
|:---|:---|
| Background: hybrid topology RBFEs | Key details about hybrid topology calculations |
| Background: the system | The MCL-1 protein and ligand series we'll be working with |
| The OpenFE CLI | Planning and running an entire campaign using the OpenFE command line interface |
| The OpenFE Python API | Building a fully custom campaign using OpenFE's Python API |
| Review & Conclusions | Recap, and where to go next |

## Background: hybrid topology binding free energies

### The relative binding free energy (RBFE) thermodynamic cycle

Rather than simulating each ligand's binding free energy in isolation (which converges very slowly), we compute the
*relative* binding free energy, ΔΔG, between pairs of similar ligands using a thermodynamic cycle:

<center> <img src="assets/rbfe_thermocycle.png" width="500"> </center>

Because this cycle is closed, we don't need to compute either of the (very hard to converge)
horizontal legs directly. Instead we simulate the vertical legs to get the relative free energy (ΔΔG).

```
ΔΔG_bind(A→B) = ΔG_bind(B) − ΔG_bind(A) = ΔG_complex(A→B) − ΔG_solvent(A→B)
```

Each vertical leg is an **alchemical transformation**: which involves simulating the
mutation of ligand A into ligand B in a given environment and then estimating the free
energy of that transformation. This means every ligand pair in an RBFE campaign requires **two** simulations: one with
the ligand transforming in solvent, and one with it transforming in the
protein-bound complex.

### Sampling along the alchemical transformation

In free energy perturbation (FEP) methods, in order to estimate the free energy of a transformation, we sample energies along the chemical transformation taking us from one end state (i.e. the first ligand) to another (i.e. the second ligand). This sampling is done along a non-physical path where we gradually turn on/off the interactions between atoms which are unique to each end state and the rest of the environment.


<br>
<center>
    <img src="assets/lambda_scaling.png" width="800">
    <figcaption align = "center"> Scaling of key parameters between the two end states across lambda. At lambda=0, the first ligand is fully interacting with the rest of the system, and the second ligand is not. At lambda=1, the first ligand is no longer interacting with the rest of the system, and the second ligand is.</figcaption>
</center>
<br>

There are many ways to sample along this "alchemical" path; a short overview can be found in the [Best Practices for Alchemical Free Energy Calculations](https://pmc.ncbi.nlm.nih.gov/articles/PMC8388617/) paper. In OpenFE, we introduce a coupling parameter, `lambda`, which interpolates the system's interactions between the two end states. We then simulate the system at a series of discrete `lambda` values, chosen so that neighbouring states have sufficient phase-space overlap. For each sampled frame, we evaluate the [reduced potential](https://www.alchemistry.org/wiki/Multistate_Bennett_Acceptance_Ratio#Reduced_potential) at every `lambda` state. These reduced potentials are then passed to the [multistate Bennett acceptance ratio (MBAR)](https://pmc.ncbi.nlm.nih.gov/articles/PMC2671659/) estimator, which yields the free energy of the transformation.

To improve sampling across lambda states, OpenFE uses a [Hamiltonian replica exchange](https://doi.org/10.1063/1.1308516) (HREX) scheme. In this approach, MD simulations of all lambda states are run simultaneously. At a fixed interval (typically every 2.5 ps), exchanges of lambda values between neighbouring replicas are attempted and accepted or rejected according to a Metropolis–Hastings criterion. This lets configurations move between lambda states, helping each replica escape local energy minima and improving sampling of the regions where adjacent states overlap. The result is a more rapidly converging free energy estimate.

<br>
<center>
    <img src="assets/hrex.jpg" width="600">
    <figcaption align = "center"> Hamiltonian replica exchange sampling along `lambda` states. Reproduced from <a href="https://pmc.ncbi.nlm.nih.gov/articles/PMC8388617/">Best Practices for Alchemical Free Energy Calculations</a> </figcaption>
</center>
<br>


### The hybrid topology scheme

There are many different ways of treating the interpolation of two end states in an alchemical simulation. Examples include single topology schemes (e.g. Transformato by Karwounopoulos et al., *Front. Mol. Biosci.*, 2022), dual coordinate / topology schemes (e.g. Separated Topologies by Baumann et al., *J. Chem. Theory Comput.*, 2023), and hybrid topology schemes as is used in OpenFE. These mostly differ in the way in which the atoms are made to appear and disappear.

As previously mentioned, OpenFE by default uses a **hybrid topology** scheme for relative free energies.
In this approach, both end-state ligands are described within a single "combined" molecule: atoms common
to both ligands (the "core") are simply interpolated, while atoms unique to either end state are grown in
or removed as dummy atoms (atoms which are non-interacting, i.e. have no nonbonded interactions with the opposite end state) over the course of the simulation. Because only the non-common atoms are perturbed,
these transformations are usually much smaller, and therefore much cheaper to converge, than perturbing complete
molecules (e.g. in absolute calculations or Separated Topologies).

<center>
    <img src="assets/hybrid_topology.png" width="500">
    <figcaption align = "center"> Hybrid Topology Scheme, reproduced from the <a href="https://www.alchemistry.org/wiki/Constructing_a_Pathway_of_Intermediate_States">alchemistry wiki</a> </figcaption>
</center>
<br>
Despite its advantages, hybrid topology calculations come with some trade-offs:

- **You need a common core.** Ligands must share a reasonable amount of chemical scaffold for a
  sensible hybrid topology mapping to exist. As an example, we want transformations that don't have too many atoms that appear/disappear, which can lead to poor convergence / variability in our binding free energy estimate. This means that you can't use this method to look
  at free energy difference between ligands with completely different chemical scaffolds. In those
  cases you would need a method like [Separated Topology](https://docs.openfree.energy/en/stable/guide/protocols/septop/septop_overview.html) instead.
- **Ligands must be aligned on their common core.** The common core must occupy very similar positions
  between the two end states. Atom mappers such as [Kartograf](https://github.com/OpenFreeEnergy/kartograf)
  inherently make this assumption and will fail to map ligand pairs which are not aligned. This is why
  our input ligands in `ligands.sdf` are all pre-aligned on a common pose.
- **Binding modes should remain similar between ligands.** Because we are assuming a common core, large binding site rearrangements can be hard to capture when transforming between ligands. Should these happen, you can end up with poor estimates of the free energy.
- **It isn't a formally exact method.** Handling dummy atoms introduces some approximations that
  can occasionally bias the free energy estimate (see [Fleck et al., *J. Chem. Theory Comput.*, 2021](https://pmc.ncbi.nlm.nih.gov/articles/PMC8280730/)
  for more information on dummy-atom artefacts in alchemical calculations). In
  practice, though, hybrid topology RBFEs are still very accurate, substantially more so than
  end-point methods like MM-GBSA, as shown in OpenFE's own large-scale benchmarking study
  ([Baumann et al., *J. Chem. Inf. Model.*, 2026](https://pubs.acs.org/jcisd8/article/66/11/6429/5185029/Large-Scale-Collaborative-Assessment-of-Binding)).

### Workflow of an RBFE campaign

In OpenFE, an RBFE campaign (i.e. calculating RBFEs over a large set of ligands of interest), usually follows this workflow:

<br>
<br>
<center>
    <img src="assets/free_energy_campaign.png" width="700">
    <figcaption align = "center"> Key components of an RBFE campaign </figcaption>
</center>
<br>
<br>

Here is a brief description of each stage:
- **Process inputs**: this involves loading relevant input files (e.g. our `ligands.sdf` and `protein.pdb`), and processing them e.g.
  adding partial charges to our ligands.
- **Create a network of mappped ligand pairs**: to run relative binding free energies, we need to know a) what transformations we should
  simulate, and b) the atom correspondence between the mutating ligands (e.g. which atoms will be appearing & disappearing). Ideally we
  attempt to simulate every single possible pair of ligands, but doing so would be prohibitively expensive. Instead we select a subset of
  possible transformations - at the very minimum enough to connect all our ligands, although sometimes we also want to add extra
  transformations for redundancy. We do this by mapping all possible ligand pairs and then assigning a score for each mapping based on
  how difficult we estimate the transformation to be.
- **Create a simulation protocol**: this stage essentially defines the details of how the transformation will be simulated, e.g. settings
  such as what force field to use and how long to run each simulation.
- **Run transformations for each ligand pair**: in this step you simulate the transformations in and gather all the system information you need to estimate a free energy.
- **Gather ΔG/ΔΔG results & other analyses**: once your simulations have completed, this step analyses your simulations and gathers
  results.

In the remainder of this notebook we will demonstrate how you can run a full RBFE campaign using either the OpenFE Command Line Interface (CLI) or the Python Application Programming Interface (API).

## Background: the system inputs

### What we need to run `openfe`

The main inputs we need for `openfe` is a [PDB](https://www.wwpdb.org/documentation/file-format-content/format33/sect1.html) file describing the receptor (protein), and an [SDF](https://en.wikipedia.org/wiki/Chemical_table_file) file with the ligands you want to calculate free energies between.

These inputs must be fully prepared (i.e. the OpenFE tooling does not do structure preparation). Amongst other things, this means that the inputs must:
1. Contain all necessary atoms and be in the right protonation state.
2. The ligands must already have 3D positions which reflect their intended right binding pose (i.e. they must be pre-docked).
3. The protein termini must be capped (i.e. ACE/NME caps or have a charged termini - H-caps as used by Maestro are not accepted).
4. Missing residues in the protein must have been modelled or appropriate caps added to deal with chain breaks.
5. The ligands must be aligned along their common core, usually through some kind of MCS docking procedure.
6. Any important crystallographic waters & ions must be present in the PDB file.
7. Non-water & ion co-factors must be extracted from the PDB and placed into a separate SDF file.

### Today's dataset: MCL-1 fragment series

Throughout this tutorial we will be working with a congeneric series of fragments binding to **MCL-1**.
This dataset originates from a study by Friberg et al., *J. Med. Chem.*, 2013 and was subsequently used as a
binding free energy benchmark in Steinbrecher et al., *J. Chem. Inf. Model.*, 2015.

We are using this example set because it involves reasonably small system sizes and simple transformations
which are easy to converge.

The dataset inputs live under `./inputs`, and include:

- `ligands.sdf`: 14 small fragment ligands, all **pre-aligned to a common binding pose** (this
  matters for hybrid topology, see above!).
- `protein.pdb`: the MCL-1 receptor. *Note: whilst this PDB file does not contain any ions or co-crystal waters, these can be handled by OpenFE.*

Let's start by looking at the ligands. To do this, we load the contents of the SDF file into [RDKit](https://www.rdkit.org/).

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import MolsToGridImage

# First we load the ligands - we remove hydrogens to make it easier to visualize
supplier = Chem.SDMolSupplier("inputs/ligands.sdf", removeHs=True)
ligand_rdmols = list(supplier)
print(f"Loaded {len(ligand_rdmols)} ligands")

# Next we generate new 2D structures - the file already contains 3D conformations
# but these are hard to depict in 2D
for mol in ligand_rdmols:
    AllChem.Compute2DCoords(mol)

# Finally, we depict the ligands in 2D
MolsToGridImage(
    ligand_rdmols,
    legends=[mol.GetProp("_Name") for mol in ligand_rdmols],
    molsPerRow=5,
    subImgSize=(220, 180),
)

These are fragment-sized modifications of one another — small substituent swaps rather than
wholesale scaffold changes — which is exactly the kind of series hybrid topology RBFE handles well.

Let's also look at what one of these ligands looks like bound to the protein:

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**

Here are the main py3dmol mouse commands:
- **left button** rotate the model
- **middle button** translate the model
- **scroll wheel** zoom in and out.

</div>

In [ ]:
# First we reload our ligand molecules to get their original 3D conformations
from rdkit import Chem

supplier = Chem.SDMolSupplier("inputs/ligands.sdf", removeHs=True)
ligand_rdmols = list(supplier)

In [ ]:
# Then we load everything into py3dmol for visualization
import py3Dmol

view = py3Dmol.view(width=600, height=400)

# Load in the protein
with open("inputs/protein.pdb") as f:
    view.addModel(f.read(), "pdb")

# Load in the ligand
example_ligand = ligand_rdmols[0]
view.addModel(Chem.MolToMolBlock(example_ligand), "mol")

# Set the protein (model 0) style to cartoon and the binding site residues to stick
# Set the ligand (model 1) style to stick
view.setStyle({"model": 0}, {"cartoon": {"colorscheme": "lightgrey"}})
view.setStyle({"model": 0, "resi": ["57-61", "80-84", "93-100"]}, {'stick': {"colorscheme": "cyan"}})
view.setStyle({"model": 1}, {"stick": {"colorscheme": "cyanCarbon"}})

view.show()

<div style="background-color:#e8f5e9; border-left: 6px solid #4caf50; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

✏️ **Exercise**


Play around with visualizing the different ligands in the set.
You can do this by changing the line `example_ligand = ligand_rdmols[0]` in the cell above to index a new entry in the list of ligands.
For example: `example_ligand = ligand_rdmols[2]` for the 3rd ligand.

</div>

## The OpenFE CLI: end-to-end RBFEs using the terminal

### Before we start: using terminal commands in a notebook

In this section of the notebook we will primarily be executing **terminal commands**.
Jupyter notebooks let you run shell commands directly from a code cell by prefixing the line with `!`. This
is how we will be accessing the OpenFE CLI for the rest of this section, everything after `!` is executed as
a terminal command, not Python.

As an example, let's use some terminal commands to poke around the files we're working with. First, let's list what's in
`./inputs`:

In [ ]:
!ls -la inputs

We can also peek inside a file without opening it in an editor, using `head` (show the first N
lines) or `cat` (show the whole file). Let's look at the start of the protein PDB file:

In [ ]:
!head -n 5 inputs/protein.pdb

### Introduction to the OpenFE CLI

OpenFE provides a command line interface (CLI) as a simple yet widely used entry point into running binding free
energy calculations:

- It automatically uses our best-practice defaults.
- It has limited customisability — if you want to change the simulation length, the force field,
  or almost anything else, you'll need the Python API (covered later in this notebook).
- Its network planning is limited to a single calculation type: hybrid topology RBFE (see the
  accompanying slides for how this compares to the other Protocols OpenFE supports).
- Despite this simplicity, it's what most of our industry partners use in practice, and it's what
  we used to run the simulations in OpenFE's own large-scale benchmarking study.

Let's start by looking at the top-level help:

In [ ]:
!openfe -h

The CLI is organised into subcommands. Each one has its own `-h`/`--help`, for example:

In [ ]:
!openfe charge-molecules -h

### The CLI RBFE workflow

Running an RBFE campaign with the CLI comes down to four commands:

1. `openfe charge-molecules`: assign partial charges to every ligand up front.
2. `openfe plan-rbfe-network`: map, score, and network the ligands, and write out one simulation input file per leg.
3. `openfe quickrun`: execute each of those simulations.
4. `openfe gather`: collect the finished results into a table of ΔG / ΔΔG values.

<br>
<br>
<center>
    <img src="assets/openfe_workflow.png" width="800">
    <figcaption align = "center"> The OpenFE CLI workflow </figcaption>
</center>
<br>
<br>

### Step 1: Adding partial charges to molecules

Conventional partial charge assignment methods (like AM1-BCC) aren't perfectly reproducible,
tiny numerical differences between runs, of order 0.03 e, are common especially on different
machines. Free energy calculations are sensitive enough that these small differences can
introduce discontinuities into our thermodynamic cycle if, say, the solvent
and complex legs of the same ligand ended up with very slightly different charges. This in
turn can accumulate to large differences in free energy estimates, as demonstrated by
[Osato et al., *J. Comput. Chem.*, 2025](https://pmc.ncbi.nlm.nih.gov/articles/PMC12079016/).

To avoid this, we assign partial charges to every ligand **once**, up front, and reuse those
same charges for every transformation that ligand appears in.

OpenFE is built on the [OpenFF toolkit](https://docs.openforcefield.org/projects/toolkit/), and
supports several partial charge methods:

- `am1bcc`: (default) computed with AmberTools, or if requested the OpenEye toolkit.
- [`am1bccelf10`](https://docs.eyesopen.com/toolkits/python/quacpactk/OEProtonClasses/OEAM1BCCELF10Charges.html): a conformer-averaged AM1-BCC (requires the OpenEye toolkit).
- [`nagl`](https://docs.openforcefield.org/projects/nagl/en/latest/): OpenFF-NAGL, a [fast graph neural network charge model which reproduces AM1BCCELF10 charges](https://pubs.acs.org/jctcce/article/22/9/4507/5087422/Developing-and-Benchmarking-Sage-2-3-0-with-the).
- [`espaloma`](https://pubs.rsc.org/sc/article/13/41/12016/786333/End-to-end-differentiable-construction-of): another neural-network charge model (requires [`espaloma-charge`](https://github.com/choderalab/espaloma-charge)).

The `openfe charge-molecules` command adds partial charges to a set of molecules. By default it
uses AM1BCC via AmberTools, but this can be controlled with a settings YAML file. We've written
one out as `partial_charge_settings.yaml`, spelling out the defaults explicitly:

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**

Partial charge **methods which require the OpenEye toolkit or Espaloma are not available** in today's tutorial. This is because the OpenEye toolkit requires a license, and espaloma requires overly large code dependencies which could not be included in today's software environment.


</div>

In [ ]:
!cat inputs/partial_charge_settings.yaml

Now let's add partial charges to the ligands in `data/ligands.sdf` using this settings file:

In [ ]:
!mkdir -p cli_output
!openfe charge-molecules -M inputs/ligands.sdf -s inputs/partial_charge_settings.yaml -o cli_output/ligands_am1bcc.sdf -n 4

Briefly explaining the flags we are using:
- The `-M` flag defines the input ligand SDF file.
- The `-s` flag defines the input settings file.
- The `-o` flag defines the output SDF file (`cli_output/ligands_am1bcc.sdf`),
  which will now contain all 14 ligands along with their atomic partial charges stored as an SDF property.
- The `-n` flag defines the number of CPU threads to use to parallelise the partial charge assignment.
As an example, we can look at the first molecule's record in the newly generated file:

In [ ]:
!head -n 48 cli_output/ligands_am1bcc.sdf

The block under `<atom.dprop.PartialCharge>` lists the per-atom partial charges for the first ligand in the file, in the same atom order as the molecule block above it. When we load this file, the partial charges will be read by OpenFE, allowing the partial charges to be re-used.

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


For more information, please look at our [CLI partial charge generation tutorial](https://docs.openfree.energy/en/latest/tutorials/charge_molecules_cli_tutorial.html)


</div>

### Step 2: Planning a network of RBFE simulations

Now that our ligands are charged, we need to actually build the set of simulation inputs that will
let us calculate relative free energies between them. This step involves:

- **Mapping** how each ligand pair could transform into one another (which atoms appear, which
  disappear, and which are common between the two).
- **Scoring** every candidate mapping, as a proxy for how hard that transformation will be to
  converge.
- Using those scores to choose a **network**, i.e. which ligand pairs we'll actually simulate.
- Creating the simulation **protocol**.
- Writing out the simulation **input files**.

All of this happens in a single command: `openfe plan-rbfe-network`.

By default, this command builds a minimal spanning network (the smallest set of edges that
connects every ligand, chosen from the best scored mappings), mapping atoms with the
[Kartograf](https://github.com/OpenFreeEnergy/kartograf) atom mapper and scoring them with the
default [LOMAP scorer](https://pmc.ncbi.nlm.nih.gov/articles/PMC3837551/), then sets up the
OpenMM-based hybrid topology RBFE protocol with its default settings.

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


Some of these choices (the mapper, the network algorithm, the partial charge method) can be
overridden with a settings YAML file passed via the `-s` flag, in the same style as
`partial_charge_settings.yaml` above. To make things simple we don't use one here. See
`openfe plan-rbfe-network -h` for the full set of options, or the
[RBFE CLI tutorial](https://docs.openfree.energy/en/stable/tutorials/rbfe_cli_tutorial.html) for a
worked example.


</div>

Let's plan our network. We pass in our pre-charged ligands (`-M` flag) and the protein (`-p` flag), and ask for a single independent simulation repeat per transformation using the `--n-protocol-repeats` flag (we will control the number of repeats later
by manually executing the simulation multiple times). Files will be written to the directory `network_setup` inside of `cli_output` (`-o` flag).

In [ ]:
!openfe plan-rbfe-network -M cli_output/ligands_am1bcc.sdf -p inputs/protein.pdb -o cli_output/network_setup --n-protocol-repeats 1

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


Notice in the output log above that the charge generation step was almost instant. This is because `openfe` detected that our ligands already carried partial charges and reused them, rather than computing AM1BCC charges from scratch.

</div>

Let's look at what was created by `plan-rbfe-network`:

In [ ]:
!ls -l cli_output/network_setup

- `ligand_network.graphml`: A file containing a [`LigandNetwork`](https://docs.openfree.energy/en/latest/guide/setup/creating_ligand_networks.html), which describes just the ligands and how they're connected.
- `network_setup.json`: A file containing an [`AlchemicalNetwork`](https://docs.openfree.energy/en/latest/guide/setup/alchemical_network_model.html). This file contains all the information for the whole campaign.
- `transformations/`: A directory with all our simulation inputs. Each JSON contains all the necessary information
  (settings, structural data, etc..) to simulate an alchemical transformation. These are the ones you usually interact with.

In [ ]:
!ls cli_output/network_setup/transformations

With 14 ligands, a minimal spanning network has 13 edges, and each edge needs both a solvent and a
complex leg, hence 26 JSON files.

Let's look at how our network looks like. To do this, we will use the visualisation tooling in the Python library [Konnektor](https://github.com/OpenFreeEnergy/konnektor) to look at the `LigandNetwork` in the `ligand_network.graphml` file. This file contains all the information about how the ligands are mapped / connected to each other.

**Note:** The `openfe` CLI also offers a `view-ligand-network` command, which allows you to interactively look at ligand networks. That command is however not compatible with Jupyter notebooks, so we don't use it here.

In [ ]:
import openfe
import konnektor

with open("cli_output/network_setup/ligand_network.graphml") as f:
    cli_ligand_network = openfe.LigandNetwork.from_graphml(f.read())

display = konnektor.draw_ligand_network(cli_ligand_network, title="MCL-1 RBFE network (CLI, minimal spanning tree)", node_size=3500)

We can also look at the mappings for each transformation. This shows the atom correspondence, with the atoms in **red** being unique (i.e. either appearing or disappearing), and the atoms in **blue** being shared common atoms that change elements, every other atom is treated as a "core" atom which is interpolated directly between the two end states.

In [ ]:
# our ligand network contains the mappings as edges
mappings = list(cli_ligand_network.edges)
mappings[0]

OpenFE also provides tooling to visualise the mappings in 3D!

The `view_3d` method displays the two ligands on the left and right and them overlapped in the middle. The coloured spheres on the ligands indicate the atom correspondence, i.e. two spheres of the same color across the left and right ligand mean that those two atoms are mapped. No coloured sphere means the atom is unique (i.e. will either be made to appear to disappear fully).

In [ ]:
mappings[0].view_3d()

<div style="background-color:#e8f5e9; border-left: 6px solid #4caf50; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

✏️ **Exercise**


Try looking at the different mappings in the network. You can do so by changing the index in the `mappings` list. E.g. `mappings[3]` will give you the fourth mapping.

</div>

### Step 3: Executing simulations

<div style="background-color:#fdecea; border-left: 6px solid #f44336; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

⚠️ **Note**


We are **not** going to run any simulations in this tutorial. Even at reduced settings, a single
RBFE leg takes far longer than we have time for here, and running 26 of them would need
substantial GPU resources.

Instead we will be providing you with precalculated data that you can load which are provided under `cli_output/results`.

**Note:** Due to storage limitations, we are not providing all the simulation outputs. The precalculated data contains all the results JSON files, but only the folders with the "simulation files" (e.g. output trajectories and auto-generated analysis PNGs) for one repeat of one transformation (`ligand_12_complex_ligand_13_complex` and `ligand_12_solvent_ligand_13_solvent`).

</div>

The CLI `quickrun` command is used to execute simulations. Calling it on one of our transformation JSON files will automatically set up the molecular system (e.g. solvating, assigning the force field, creating the OpenMM inputs), run the simulation (usually requiring a GPU), and some limited post simulation analysis (e.g. MBAR matrix overlap, forward & reverse energy convergence analysis, protein backbone RMSD).

Here is an example of how you would call `quickrun` to run a single simulation:

```bash
openfe quickrun cli_output/network_setup/transformations/rbfe_ligand_1_solvent_ligand_2_solvent.json \
    -o results/rbfe_ligand_1_solvent_ligand_2_solvent.json \
    -d results/rbfe_ligand_1_solvent_ligand_2_solvent/
```

The `-o` flag indicates the file where the final output results file should be written. This is a JSON formatted file that contains key information about the simulation results such as the binding free energy estimate and the uncertainty.

The `-d` flag indicates where the simulation output files should be written, including the trajectory files, output PNGs of the automated analyses, etc...

To run multiple repeats of the same simulation (we recommend running at least 3 repeats to get an estimate of the sampling errror), you would call `openfe quickrun` on the same input multiple times but giving it different paths for `-o` and `-d`.

For example you could do the following to run three repeats where each repeat has its outputs written to a different `repeat` subfolder (`repeat0`, `repeat1`, and `repeat2`):

```bash
openfe quickrun cli_output/network_setup/transformations/rbfe_ligand_1_solvent_ligand_2_solvent.json \
    -o results/repeat0/rbfe_ligand_1_solvent_ligand_2_solvent.json \
    -d results/repeat0/rbfe_ligand_1_solvent_ligand_2_solvent/
```

```bash
openfe quickrun cli_output/network_setup/transformations/rbfe_ligand_1_solvent_ligand_2_solvent.json \
    -o results/repeat1/rbfe_ligand_1_solvent_ligand_2_solvent.json \
    -d results/repeat1/rbfe_ligand_1_solvent_ligand_2_solvent/
```

```bash
openfe quickrun cli_output/network_setup/transformations/rbfe_ligand_1_solvent_ligand_2_solvent.json \
    -o results/repeat2/rbfe_ligand_1_solvent_ligand_2_solvent.json \
    -d results/repeat2/rbfe_ligand_1_solvent_ligand_2_solvent/
```

This is what we did when generating results for this tutorial.

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


See the [RBFE CLI tutorial](https://docs.openfree.energy/en/stable/tutorials/rbfe_cli_tutorial.html#run-the-simulations)
for more detail on running simulations, including a template for submitting simulations on a HPC machine.


</div>

An example output can be found under `cli_output/results/repeat0`. There we can look at the "rbfe_ligand_12_complex_ligand_13_complex" transformation.

In [ ]:
!ls -ld cli_output/results/repeat0/rbfe_ligand_12_complex_ligand_13_complex*

As we can see, we have both a JSON file with our results and a directory where some of the simulation files are.

Looking inside the simulation files directory, you can see that we have:
- A **"shared_HybridTopologySetupUnit"** directory, which contains artifacts from how the system was set up, including an XML
  of the OpenMM system we created for simulation.
- A **"shared_HybridTopologyMultiStateSimulationUnit"** directory, which contains our simulation outputs, particularly our NetCDF trajectory `simulation.nc`.
- A **"shared_HybridTopologyMultiStateAnalysisUnit"** directory, which contains some of our automated analysis artifacts.

**Note:** some of our file names and directories contain a long alphanumeric string like "d45cad707529440a89a9e4e4b1853aeb", this is a unique identifier which openfe uses internally to track the simulation being executed.

**Note 2:** you will notice the word **"attempt_0"** in the directory names. By default `quickrun` will attempt to run a simulation up to 3 times if it fails. If that happens, the attempt number will increase, e.g. **"attempt_1"**.

In [ ]:
!ls cli_output/results/repeat0/rbfe_ligand_12_complex_ligand_13_complex/*

We can briefly look at some of the analysis PNGs we have created for this transformation.

First let's look at the MBAR overlap matrix. This output is generated by the [MBAR free energy esimator](https://github.com/choderalab/pymbar) and tells us how well the different lambda states we simulated (see the [introduction](#Sampling-along-the-alchemical-transformation) for more details) overlapped energetically. A value of less than 0.03 in the direct off diagonal would show poor overlap and therefore poor convergence.

In [ ]:
from IPython.display import Image

basepath="cli_output/results/repeat0/rbfe_ligand_12_complex_ligand_13_complex"

Image(filename=f"{basepath}/shared_HybridTopologyMultiStateAnalysisUnit-d45cad707529440a89a9e4e4b1853aeb_attempt_0/mbar_overlap_matrix.png")

Next we look at the ligand center-of-mass drift. This tells us how much the ligand has moved from its initial pose at the start of the production simulation. It offers a proxy for how dynamic the simulation was, large values (greater than 5 Angstrom) indicate a possibility that the ligand has moved out of the binding site.

In [ ]:
Image(filename=f"{basepath}/shared_HybridTopologyMultiStateAnalysisUnit-d45cad707529440a89a9e4e4b1853aeb_attempt_0/ligand_COM_drift.png")

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


You can find out more about the types of analyses the Hybrid Topology Protocol generates in the [OpenFE user guide](https://docs.openfree.energy/en/stable/guide/protocols/relativehybridtopology.html#analysis).


</div>

### Gathering results

Now that we have run all our simulations, we can gather our free energy results. This is done using the `openfe gather` command.

First let's start by gathering the "raw" results, that is the ΔG for every individual leg and repeat. All we have to do is pass the results JSON files to `openfe gather` with the `--report raw` flag:

In [ ]:
!openfe gather --report raw cli_output/results/repeat?/*json

Next, let's look at the ΔΔG results, where we combine both legs of the thermodynamic cycle and average our results over the repeats.

**Note:** because we have multiple repeats, the uncertainty we report is the standard deviation in the estimate between repeats.

This can be done with the `--report ddg` flag:

In [ ]:
!openfe gather --report ddg cli_output/results/repeat?/*json

Finally, in some cases, it is useful to have an estimate for the absolute ΔG per ligand. This is particularly useful when you want to rank your compounds, for example to understand which compound should be prioritised for synthesis, or to see how well your predictions recover experimental rankings. In OpenFE, we can use a maximum likelihood estimator to generate these values from the ΔΔG values. This can be done with the `--report dG` flag.

**Important:** these absolute values are not true absolutes, instead they are relative to the mean of the ligands which is set to a ΔG value of 0 kcal/mol. If you want to compare these to experiment, you would need to shift them by the experimental value.

In [ ]:
!openfe gather --report dg cli_output/results/repeat?/*json

In the API portion of this tutorial, we will demonstrate how you can go about plotting these results using our tool [Cinnabar](https://github.com/OpenFreeEnergy/cinnabar).

## The OpenFE Python API: running custom RBFE simulations

The CLI is convenient, but sometimes you need something it can't give you — a longer simulation,
a different force field, a non-default network topology. For that, we turn to the Python API.

In this section we will set up a similar campaign to the one we created with the CLI, but with
a few deliberate changes to demonstrate how you can use the API to customise simulations:

- A **redundant minimal spanning tree** network instead of a minimum spanning tree, giving us
  some redundancy to fall back on if an edge fails.
- **10 ns** of production simulation instead of the CLI's default 5 ns.
- **AshGC** partial charges together with the newest **OpenFF 2.3.0** force field
  (AshGC is the charge model OpenFF 2.3.0 was fitted against), instead of the default OpenFF 2.2.1 and AM1BCC charges.

### Step 1: Loading molecules

To create our simulations, we need objects which describe every part of the system.

OpenFE represents these as **`Component`s**; different portions of the system that
can be put composed together into a full chemical system (we'll meet the `ChemicalSystem` class itself shortly).

The first `Component` we will introduce is the `SmallMoleculeComponent`, which is geared specifically at holding
small molecules.

Let's reload our `ligands.sdf` file and create `SmallMoleculeComponent`s for every ligand:

In [ ]:
# Execute this cell to remove some of the excess warning text we get
import warnings
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow")

In [ ]:
import openfe
from rdkit import Chem

# Let's read our SDF file again
supplier = Chem.SDMolSupplier("inputs/ligands.sdf", removeHs=False)
ligand_rdmols = list(supplier)

# We then load every RDKit Molecule into a SmallMoleculeComponent.
ligands = [openfe.SmallMoleculeComponent.from_rdkit(mol) for mol in ligand_rdmols]

The main entry point for `SmallMoleculeComponent`s is an RDKit Molecule (in fact this is the main way we represent nearly all our molecules internally!). This is because RDKit Molecules include all the bond order and element information that we would need when doing operations such as mapping a transformation or assigning force field parameters.

`SmallMoleculeComponent`s, like some of our other `Components` have some built-in utilities. For example, you can get the total charge by calling the `total_charge` property:

In [ ]:
ligands[0].total_charge

To load the protein we use a `ProteinComponent`, which is made specifically for handling biopolymers.

With the `ProteinComponent`, we can directly load the PDB rather than going through a secondary object type. Our PDB parser is inherited from [OpenMM](https://openmm.org/), so we can load in any type of system that they can:

In [ ]:
protein = openfe.ProteinComponent.from_pdb_file("inputs/protein.pdb")

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


Whilst our current PDB file only contains the protein, the `ProteinComponent` does support loading in crystallographic waters and ions. However it cannot handle loading in non-standard residues or small molecules.


</div>

`ProteinComponent` also has some useful utilities, for example you can convert them easily to OpenMM Topologies:

In [ ]:
topology = protein.to_openmm_topology()
topology

Or get their position arrays in a format OpenMM will accept:

In [ ]:
positions = protein.to_openmm_positions()
print("number of particle positions: ", len(positions))
print("positions of first 5 atoms: ", positions[:5])

Finally, we have to define how our system will be solvated. To do this we use a `SolventComponent`. This object doesn't hold the positions of the water, but rather information about its composition.

For example here we explicitly build the default configuration for a `SolventComponent`, defining the solvent as water (`smiles='O'`), 'Cl-' as the negative counterion, 'Na+' as the positive counterion, a target ionic concentration of 0.15 M, and that the system will be neutralized:

In [ ]:
# OpenFE uses OpenFF units when defining quantities
from openff.units import unit

solvent = openfe.SolventComponent(smiles='O', negative_ion='Cl-', positive_ion='Na+', ion_concentration=0.15 * unit.molar, neutralize=True)

This information will be used when we execute our transformation to build the system according to those specifications.

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


If you already have a pre-solvated, pre-equilibrated system (for example, one that includes a
membrane), you can use `SolvatedPDBComponent` or `ProteinMembraneComponent` instead of a separate
`ProteinComponent` + `SolventComponent` pair. See the
[OpenFE documentation](https://docs.openfree.energy/en/stable/) for more on preparing
membrane-containing systems.


</div>

### Step 2: Generating partial charges

Just like `openfe charge-molecules` on the CLI, the Python API has a function for bulk-assigning
partial charges: `bulk_assign_partial_charges`. Here we will use it to assign [AshGC charges, a
graph neural network charge model designed to reproduce AM1BCCELF10 charges quickly](https://pmc.ncbi.nlm.nih.gov/articles/PMC13173527/).
The AshGC model is distributed as the [OpenFF NAGL](https://github.com/openforcefield/openff-nagl) model `openff-gnn-am1bcc-1.0.0.pt`.

Here we use AshGC charges not only because they are high quality but also because this was the charge model OpenFF 2.3.0 (the force field we will be using) was fitted for.

In [ ]:
from openfe.protocols.openmm_utils.charge_generation import bulk_assign_partial_charges

ligands = bulk_assign_partial_charges(
    molecules=ligands,
    overwrite=False,
    method="nagl",
    toolkit_backend="ambertools",
    generate_n_conformers=None,
    nagl_model="openff-gnn-am1bcc-1.0.0.pt",
    processors=4,
)

As you can see AshGC charges are applied very quickly, much quicker than when we obtained AM1BCC charges with AmberTools earlier. This is one of the main advantages of AshGC charges; they avoid needing semi-empirical QM calculations, which can be very time consuming especially for larger molecules.

We can confirm the charges are attached by converting one of our SmallMoleculeComponents to an OpenFF `Molecule` and
inspecting its `partial_charges`:

In [ ]:
offmol = ligands[0].to_openff()
offmol.partial_charges

### Step 3: Defining a network of ligand transformations

As mentioned earlier, in principle we could try to compute the free energy between
every possible pair of our 14 ligands. In practice, we don't want to because:

1. It scales as O(n²) in the number of ligands, which is prohibitively expensive once
   you have more than a handful.
3. Not every transformation would even succeed. Some ligand pairs are simply too different
   for a good hybrid topology mapping to exist. This is especially true for more complex
   ligand series where the transformations sometimes differ by more than just an R group
   transformation, or where the protein adapts around the ligands of interest.

Instead, we need to select a subset of transformations that we need to run. At the very minimum
this is a network that keeps every ligand **connected** to every other ligand, even if only indirectly.
This is required so that we can effectively calculate the relative free energy difference between any
two pairs of ligands in our series, which is possible even if we must calculate it through an indirect
route through the transformation of another ligand.

Selecting our network is a three-step process:
1. Choose an **atom mapper** that will be used to obtain the atom correspondence between our ligand pairs.
2. Choose a **scorer** for our atom mappings that will estimate the difficulty of the transformation.
3. Choose a **network planning algorithm** that will use the mapper and scorer to select desired transformations.

In the CLI, all three of these was automatically done by the `plan-rbfe-network` command.

#### Mapping two ligands

As mentioned earlier, in a hybrid topology RBFE, an atom mapping tells us which atoms are unique to each end-state
ligand (and so will appear or disappear over the simulation) and which are common to both (and so
will simply be interpolated between the two states).

In the CLI example above, this was done automatically with the [Kartograf](https://github.com/OpenFreeEnergy/kartograf) atom mapper. Kartograf is OpenFE's 3D-first mapper, which uses the ligands' 3D coordinates to build a mapping based on how close similar atoms are.
Let's use it directly via the Python API on a single pair of ligands:

In [ ]:
mapper = openfe.KartografAtomMapper()

# suggest_mappings returns a generator; Kartograf normally suggests a single mapping per pair
mapping = next(mapper.suggest_mappings(ligands[0], ligands[1]))
mapping

As a reminder, in our 2D descriptions, atoms unique to one ligand or the other (i.e. atoms that will appear/disappear) are highlighted in red and atoms that are mapped but change element are shown in blue. Atoms without a colour are mapped between the two ligands.

As we did earlier, we can also look at the same mapping in 3D:

In [ ]:
mapping.view_3d()

At its core, an `AtomMapping` object (which is what `mapping` contains) essentially holds a reference to the two ligands and how they are mapped. This can be accessed as a dictionary when accessing `componentA_to_componentB`, where the keys are the atom indices of the first ligand, and the values the indices of the corresponding atoms in the second ligand. Any indices not in this dictionary are unique to their respective ligand.

In [ ]:
mapping.componentA_to_componentB

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


See the [Creating Atom Mapping](https://docs.openfree.energy/en/stable/guide/setup/creating_atom_mappings_and_scores.html) entry of the OpenFE user guide for more information on atom mappers.

</div>

<div style="background-color:#e8f5e9; border-left: 6px solid #4caf50; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

✏️ **Exercise**


OpenFE also ships a second atom mapper, `openfe.LomapAtomMapper`, which works by maximum common
substructure (MCS) search rather than 3D geometry.

Try mapping the same pair of ligands (`ligands[0]` and `ligands[1]`) with `LomapAtomMapper()`
instead of `KartografAtomMapper()`, and compare the resulting mapping to the Kartograf one above.
Do they map the same atoms? Why would they be different?

**Hint:** the `KartografAtomMapper` has automatically applied built-in [rules for filtering](https://kartograf.openfree.energy/en/latest/api/kartograf.filters.html) out some types of transformations which are deemed to usually cause issues.


</div>

#### Scoring two ligands

To decide which ligand pairs are worth simulating, we need some way to estimate up front how
*hard* a given transformation will be. This is done using an atom mapping **scorer**. A scorer
takes a `LigandAtomMapping` and returns a score from 0 (worst) to 1 (best).

Here we will use the LOMAP scorer (default scorer in OpenFE), which applies a set of empirical rules that penalise
transformations known to be difficult. For example, the LOMAP scorer penalises transformations
that involve large amounts of appearing/disappearing atoms or situations where the transformation
involves breaking open a ring. The full list of scorer rules can be found in the [LOMAP paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC3837551/).

Let's use the LOMAP scorer to score the mapping we created earlier:

In [ ]:
score = openfe.lomap_scorers.default_lomap_score(mapping)
score

The score of ~0.9 indicates that this is likely an easy transformation. The scorer is empirical and sometimes poorly estimates difficulty, but on average we find that it generally ends up being a good initial estimate of difficulty.

<div style="background-color:#e8f5e9; border-left: 6px solid #4caf50; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

✏️ **Exercise**


`openfe.lomap_scorers` has several other scoring functions beyond `default_lomap_score`, for
example `mcsr_score` (maximum common substructure size), `atomic_number_score`, and
`hybridization_score`. These are combined together in the `default_lomap_score`.

Try scoring the same `mapping` with one or two of these scoring functions. Do they
agree with `default_lomap_score` about how easy this transformation is?

**Hint:** you have to import the `lomap_scorers` module before you can use these scorers.

</div>

#### Creating ligand transformation networks

Finally, now that we have a mapper and a scorer, we need to generate the network of transformations
we want to simulate. We do this by creating a `LigandNetwork`.

In the CLI example, `plan-rbfe-network` automatically built a minimal spanning tree (MST), i.e. the
network with the fewest possible transformations that still connects every ligand. Here, we will instead
build a **redundant** MST, which ensures that each ligand is connected to at least two ligands (instead of one).

Building a `LigandNetwork` just requires the mapper and scorer we defined above, plus the ligands
we want in the network:

In [ ]:
mapper = openfe.KartografAtomMapper()
scorer = openfe.lomap_scorers.default_lomap_score

ligand_network = openfe.ligand_network_planning.generate_minimal_redundant_network(
    ligands=ligands,
    mappers=[mapper],
    scorer=scorer,
    mst_num=2,  # make sure each ligand is at least connected to 2 other ligands
)

print(f"{len(ligand_network.nodes)} ligands, {len(ligand_network.edges)} edges")

Each edge in the network gives you back the two ligands involved and the mapping between them:

In [ ]:
an_edge = next(iter(ligand_network.edges))
print("mapping: ", an_edge.componentA_to_componentB)
an_edge

<div style="background-color:#e8f5e9; border-left: 6px solid #4caf50; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

✏️ **Exercise**


Why would you choose to have a **redundant MST** instead of just simulating the minimum number
of edges possible?

</div>

Let's visualise the resulting network with konnektor, first as a static plot:

In [ ]:
import konnektor

view = konnektor.draw_ligand_network(ligand_network, title="MCL-1 RBFE network (API, redundant MST)", node_size=5000)

Since the network is complicated, we also have an interactive widget where you can more easily see the transformations.

Note: you can zoom in and drag parts of the network around to see the connections.

In [ ]:
from konnektor.visualization.widget import draw_network_widget

view = draw_network_widget(ligand_network, represent_molecules_twoD=True)

<div style="background-color:#e8f5e9; border-left: 6px solid #4caf50; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

✏️ **Exercise**


Try building a couple of the other network topologies OpenFE offers, using the same `ligands`,
`mapper` and `scorer`:

- `openfe.ligand_network_planning.generate_minimal_spanning_network()`: a plain, non-redundant
  MST, this is what the CLI used above.
- `openfe.ligand_network_planning.generate_radial_network()`: a star network around one chosen ligand.
- `openfe.ligand_network_planning.generate_maximal_network()`: this just tries to connect every ligand to every other ligand.

Visualise each one with the `konnektor` utilities we demonstrated above.

In what situation would you want to run one network type over another?


</div>

### Step 4: Creating a `Protocol`

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


A `Protocol` is OpenFE's description of *how* to compute a free energy difference between two
end states (i.e. `ChemicalSystem`s). This includes; the sampling method, force field, simulation length,
and everything else needed to actually run and analyse the calculation. You can look at the [Protocol user guide entry](https://docs.openfree.energy/en/stable/guide/setup/defining_protocols.html) and the
[Choose and Configure a Protocol cookbook](https://docs.openfree.energy/en/stable/cookbook/choose_protocol.html) for more information, if needed.

</div>

The `RelativeHybridTopologyProtocol` is the OpenMM-based hybrid topology `Protocol` we have been using
throughout this tutorial.

The main input to a `Protocol` are settings, these will dictate how the `Protocol` will behave. We can
obtain a set of default settings for the `RelativeHybridTopologyProtocol` by calling its `default_settings` method:

In [ ]:
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol

settings = RelativeHybridTopologyProtocol.default_settings()

This `settings` object contains the following sub-settings:
1. `alchemical_settings`: settings controlling how the alchemical simulation will run.
2. `engine_settings`: settings contolling how the MD engine will be executed.
3. `forcefield_settings`: what force fields details to use.
4. `integrator_settings`: settings controlling the MD simulation integrator.
5. `lambda_settings`: settings controlling the number of lambda windows and how the parameters will be interpolated across them.
6. `output_settings`: settings controlling the files that will be written to disk.
7. `partial_charge_settings`: settings controlling how the partial charges will be applied to the small molecules **if** we haven't already assigned charges (i.e. it is generally ignored).
8. `simulation_settings`: settings controlling the simulation, e.g. the simulation lengths.
9. `solvation_settings`: settings controlling how the system will be solvated when built.
10. `thermo_settings`: settings for some of the thermodynamic parameters, e.g. temperature.

Let's look at what each of these contain:

In [ ]:
settings.alchemical_settings

In [ ]:
settings.engine_settings

In [ ]:
settings.forcefield_settings

In [ ]:
settings.integrator_settings

In [ ]:
settings.lambda_settings

In [ ]:
settings.output_settings

In [ ]:
settings.partial_charge_settings

In [ ]:
settings.simulation_settings

In [ ]:
settings.solvation_settings

In [ ]:
settings.thermo_settings

As you can see, there are quite a few things you can change, from how long the simulation will run (as defined in `simulation_settings`), to more obscure settings such as the softcore potential used in our alchemical transformation (as defined in `alchemical_settings`).

As mentioned earlier, in this exercise we want to:
- Run a longer production simulation (10 ns instead of the default 5 ns).
- Use the new OpenFF 2.3.0 force field for our ligand (instead of the default of OpenFF 2.2.1).

Here is how we can do this:

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**

OpenFE uses [OpenFF units](https://docs.openforcefield.org/projects/units/en/stable/), a derivative of [Pint](https://pint.readthedocs.io/en/stable/index.html), to define quantities such as time.

</div>

In [ ]:
from openff.units import unit

# 10 ns of production instead of the default 5 ns
settings.simulation_settings.production_length = 10.0 * unit.nanosecond

# the newest force field, matched to the AshGC charges we generated above
settings.forcefield_settings.small_molecule_forcefield = "openff-2.3.0"

# we also set it to a single repeat per transformation to replicate the CLI's --n-protocol-repeats
# we can then execute the protocol multiple times to get repeats
settings.protocol_repeats = 1

The settings automatically validate the input and will raise an error if you attempt to pass through the wrong thing. For example, this is what happens if you try to pass a unit of length to a setting that expects time:

In [ ]:
settings.simulation_settings.equilibration_length = 10.0 * unit.angstrom

Next we need to adapt our settings to the type of transformation we will be doing since
solvent and complex transformations usually need slightly different settings. For
example, the complex leg doesn't need as much water padding around it, since most of the box is
already filled by the protein.

Rather than set this by hand, we can reuse the same helper the CLI
uses internally, `RelativeHybridTopologyProtocol._adaptive_settings`, which adapts a settings
object based on the `ChemicalSystem`s and mapping it will be used for.

<div style="background-color:#fdecea; border-left: 6px solid #f44336; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

⚠️ **Note**


Here we create a single set of complex and solvent settings & Protocols for all our ligand transformations.
This is because all the transformations are very similar. If they were to change substantially, e.g. if some
involve net charge changes and others did not, you would need to create a set of Protocols per transformation type.

</div>

First, let's start by defining a `ChemicalSystem` for an example transformation.

A `ChemicalSystem` is a container that holds all the `Component`s that define one of the end states.
For a complex transformation, this usually holds:
- A `SmallMoleculeComponent`: the ligand that is transforming.
- A `ProteinComponent`: the protein the ligand is bound to.
- A `SolventComponent`: how the system will be solvated.

For a solvent transformation, it would be the same except without the `ProteinComponent`.

With this information, the `Protocol` can then not only understand how to build the system but also identify which parts of the system are changing.

Now let's create some ChemicalSystems for the complex & solvent transformations for the first edge in our ligand network:

In [ ]:
# Extract the first mapping in the network
mapping = next(iter(ligand_network.edges))

# We can then construct the complex ChemicalSystems
stateA_complex = openfe.ChemicalSystem(
    {
        "ligand": mapping.componentA, # This is the SmallMoleculeComponent stored as state A in the mapping
        "protein": protein, # This is the ProteinComponent we loaded much earlier in this tutorial
        "solvent": solvent, # Similarly this is the SolventComponent we created a while back
    }
)

stateB_complex = openfe.ChemicalSystem(
    {
        "ligand": mapping.componentB, # This is the only thing that changes from stateA_complex
        "protein": protein,
        "solvent": solvent,
    }
)

# Now the solvent ChemicalSystems - the only thing that changes is the lack of ProteinComponent
stateA_solvent = openfe.ChemicalSystem(
    {
        "ligand": mapping.componentA,
        "solvent": solvent,
    }
)

stateB_solvent = openfe.ChemicalSystem(
    {
        "ligand": mapping.componentB,
        "solvent": solvent,
    }
)

Now we can use these `ChemicalSystem`s to adapt our settings for the type of solvent & complex transformations that we have in our network:

In [ ]:
complex_settings = RelativeHybridTopologyProtocol._adaptive_settings(
    stateA=stateA_complex,
    stateB=stateB_complex,
    mapping=mapping,
    initial_settings=settings, # These are the settings we manually amended earlier
)

solvent_settings = RelativeHybridTopologyProtocol._adaptive_settings(
    stateA=stateA_solvent,
    stateB=stateB_solvent,
    mapping=mapping,
    initial_settings=settings,
)

# Let's look at what happens to our solvent padding
print("solvent leg padding:", solvent_settings.solvation_settings.solvent_padding)
print("complex leg padding:", complex_settings.solvation_settings.solvent_padding)

Finally, with settings for both our solvent and complex transformations, let's create our two `Protocol`s:

In [ ]:
solvent_protocol = RelativeHybridTopologyProtocol(solvent_settings)
complex_protocol = RelativeHybridTopologyProtocol(complex_settings)

### Step 5: Creating `Transformation`s

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


A `Transformation` ties everything together into a single "simulation ready" object:
- Two `ChemicalSystem`s defining the start and end states.
- An `AtomMapping` defining the atom correspondence.
- The `Protocol` defining the type of simulation and its settings.

It's the object we eventually write to file and hand to `openfe quickrun`.

</div>

Just as with the CLI, every ligand pair (every edge in our network) needs **two**
`Transformation`s, one for the solvent leg, one for the complex leg.

Let's demonstrate this for a single ligand pair first, before doing it for the whole network. We will re-use the mapping & `ChemicalSystem`s we built in the previous section:

In [ ]:
solvent_transformation = openfe.Transformation(
    stateA_solvent,
    stateB_solvent,
    solvent_protocol,
    mapping=mapping,
    # It's usually a good idea to give it a descriptive name so that we know what the transformation is
    name=f"rbfe_{mapping.componentA.name}_solvent_{mapping.componentB.name}_solvent",
)

complex_transformation = openfe.Transformation(
    stateA_complex,
    stateB_complex,
    complex_protocol,
    mapping=mapping,
    # It's usually a good idea to give it a descriptive name so that we know what the transformation is
    name=f"rbfe_{mapping.componentA.name}_complex_{mapping.componentB.name}_complex",
)

# Let's print out the names of each Transformation
print("solvent: ", solvent_transformation.name, "complex: ", complex_transformation.name)

Now let's do this for every edge in the network, creating a list of `Transformation`s:

In [ ]:
# We create a dictionary that holds the different Components that
# we need to use for each type of transformation
legs = {
    "solvent": {"solvent": solvent},
    "complex": {"solvent": solvent, "protein": protein},
}

# Similarly, this dictionary holds the protocol for each leg
protocols = {"solvent": solvent_protocol, "complex": complex_protocol}

transformations = []
for mapping in ligand_network.edges:
    for leg, shared_components in legs.items():
        # We use ** to unpack the contents of the dictionary
        stateA = openfe.ChemicalSystem({"ligand": mapping.componentA, **shared_components})
        stateB = openfe.ChemicalSystem({"ligand": mapping.componentB, **shared_components})
        name = f"rbfe_{mapping.componentA.name}_{leg}_{mapping.componentB.name}_{leg}"
        transformations.append(
            openfe.Transformation(stateA, stateB, protocols[leg], mapping=mapping, name=name)
        )

print(f"{len(transformations)} transformations")

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


Whilst not used here, you can also create an `AlchemicalNetwork` by passing it a list of `Transformation`s.
This offers the ability to explore your `Transformation`s in a single object and is also the starting point
for more advanced functionality in OpenFE tooling.

</div>

### Step 6: Writing `Transformation`s to file

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


Nearly every OpenFE object (`LigandNetwork`, `Transformation`, `AlchemicalNetwork`, `Protocol`, and more) can be written
to a JSON file with `.to_json()` and read back with `.from_json()`.

</div>

To actually run our campaign, we need to write each `Transformation` out to its own JSON file, so
that it can be picked up by `openfe quickrun`:

In [ ]:
import pathlib

# We will write our transformations inside of a dictionary named "transformations" inside of "api_outputs"
transformation_dir = pathlib.Path("api_output/transformations")
transformation_dir.mkdir(exist_ok=True, parents=True)

# Finally we write out all the transformations based on their names
for transformation in transformations:
    transformation.to_json(transformation_dir / f"{transformation.name}.json")

!ls api_output/transformations

### Step 7: Executing our simulations

<div style="background-color:#fdecea; border-left: 6px solid #f44336; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

⚠️ **Note**


As with the CLI section, we won't run any of these simulations here, 52 transformations
(26 edges × 2 legs) at 10 ns of production each is well beyond what we can do in a tutorial
session. Precalculated results are present under `api_output/results`.

**Note:** Due to storage limitations, we are not providing all the simulation outputs. The precalculated data contains all the results JSON files, but only the folders with the "simulation files" (e.g. output trajectories and auto-generated analysis PNGs) for one repeat of one transformation (`ligand_12_complex_ligand_13_complex` and `ligand_12_solvent_ligand_13_solvent`).

</div>

While it is possible to manually execute simulations via the Python API, these are complicated and overly clunky. So we recommend using the `quickrun` CLI command to do this instead, like we did [earlier](#Step-3:-Executing-simulations).

This is what we did here, we ran `quickrun` on all our inputs in triplicate. Results for each repeat are found in `api_output/results/repeat0`, `api_output/results/repeat1`, and `api_output/results/repeat2`.

### Step 8: Gathering free energy results

Similar to [simulation execution](#Executing-our-simulations), the best way to gather free energy results across all our simulation outputs is using the `gather` CLI command.

Let's try it out here:

In [ ]:
!openfe gather api_output/results/repeat?/*json --report ddg

<div style="background-color:#e8f5e9; border-left: 6px solid #4caf50; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

✏️ **Exercise**


Try using `openfe gather` to get the `dg` and `raw` free energy results for this new set of simulations.


</div>

### Step 9: Exploring & plotting our free energy results using cinnabar

Beyond extracting free energy results, we can also use the Python package [Cinnabar](https://cinnabar.openfree.energy/en/latest/) to explore our free energy results.

Cinnabar offers a wealth of utilities specifically geared towards analysing networks of free energy results. For example, the maximum likelihood estimator that we use to calculate ΔG results from our ΔΔG results (i.e. when we call `openfe gather --report dg`) uses Cinnabar in the background. Other utilities include; different means of obtaining statistics when comparing against experiment (e.g. deviation and correlation metrics), plotting utilities, and cycle closure error analysis.

In this section we will demonstrate some of these utilities.

#### Loading data into Cinnabar

The two main inputs that Cinnabar needs are:
1. Calculated ΔΔG results for all ligand pairs we want to analyze.
2. (optional) ΔG experimental results to compare against our calculated results.

To load our **calculated results**, we first start by using `openfe gather` and storing the output to a file by using the `-o` flag:

In [ ]:
!openfe gather api_output/results/repeat?/*json --report ddg -o api_ddG_results.tsv

This command wrote a tab-separated values (TSV) file named `api_ddG_results.tsv` with all the ΔΔG results.

We can now load it back into Python for further manipulation by reading the file into a [pandas](https://pandas.pydata.org/) dataframe:

In [ ]:
import pandas as pd

results = pd.read_csv("api_ddG_results.tsv", sep="\t")
results.head()

We also have **experimental results** in a TSV named `experimental_data.tsv` which we can also load into a pandas dataframe.

In [ ]:
exp_data = pd.read_csv("experimental_data.tsv", sep="\t")
exp_data.head()

<div style="background-color:#fdecea; border-left: 6px solid #f44336; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

⚠️ **Note**


The experimental values have an **uncertainty of 0.0 kcal/mol** because the original publication did not report uncertainties in their affinity measurements.

</div>

We can now load our results into Cinnabar. We do this by creating an `FEMap`, this the main object in cinnabar which holds all the calculated and experimental values as well as provides an entry point for built-in analyses and plotting methods.

In [ ]:
from cinnabar import FEMap

fe_map = FEMap()

Let's start by feeding in our calculated values. We do so by calling the `add_relative_calculation` method for each row in our results dataframe:

In [ ]:
from openff.units import unit

for index, row in results.iterrows():
    fe_map.add_relative_calculation(
        labelA=row['ligand_i'],  # This is a unique label for the first ligand
        labelB=row['ligand_j'],  # This is a unique label for the second ligand
        # we have to add the units - we use openff.units in openfe
        value=row['DDG(i->j) (kcal/mol)'] * unit.kilocalories_per_mole,  # The binding free energy estimate
        uncertainty=row["uncertainty (kcal/mol)"] * unit.kilocalories_per_mole,  # The uncertainty in the estimate
        source="openfe",  # A tag to identify where this calculated data came from
    )

Next we add the experimental data using `add_experimental_measurement`:

In [ ]:
for index, row in exp_data.iterrows():
    fe_map.add_experimental_measurement(
        label=row['ligand'],
        value=row['DG (kcal/mol)'] * unit.kilocalories_per_mole,  # The absolute affinity of the ligand
        uncertainty=row["uncertainty (kcal/mol)"] * unit.kilocalories_per_mole,
        source="experimental",
    )

We now have all the information we need inside of `fe_map`. We can briefly have a look at the information contained in the object using the `draw_graph()` method. This will plot a network with all the nodes (ligands) and edges (relative calculations) information we pass through:

In [ ]:
fe_map.draw_graph()

#### Comparing relative free energies (ΔΔG) to experiment

The Cinnabar `FEMap` automatically generates relative free energies from all the experimental and calculated results that we passed through. We can access these results as a [pandas Dataframe](https://pandas.pydata.org/docs/user_guide/dsintro.html#dataframe) using the `get_relative_dataframe()` method of the `FEMap` object:

In [ ]:
dataframe = fe_map.get_relative_dataframe()

In [ ]:
# Let's look at some of the experimental data sources
dataframe[dataframe['source'] == 'experimental'].head()

In [ ]:
# Let's also look at some of the openfe calculated data sources
dataframe[dataframe['source'] == 'openfe'].head()

Cinnabar offers some convenient tools for plotting our relative free energy results against experiment. We can do this by passing `fe_map` to the `plot_DDGs` helper method:

In [ ]:
from cinnabar import plotting

plotting.plot_DDGs(fe_map, source="openfe", figsize=5, title="Relative Free Energies")

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


By default, the Cinnabar plotting utilities automatically calculate some statistics when plotting.
For ΔΔG, this is limited to deviation metrics (i.e. RMSE and MUE). This is because correlation statistics, e.g. Pearson Rho and Kendall tau, are often innapropriate when comparing ΔΔG plots as the values can be influenced by the symmetry in the relative transformation.

</div>

<div style="background-color:#e8f5e9; border-left: 6px solid #4caf50; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

✏️ **Exercise**


Try playing around with the `map_positive` and `symmetrise` arguments of `plot_DDGs`, for example: `plotting.plot_DDGs(fe_map, source="openfe", symmetrise=True)`.

What do they do? Why could it be useful to use these?


</div>

#### Comparing absolute free energies (ΔG) to experiment

As previously mentioned, Cinnabar can convert relative free energy estimates (ΔΔG) to absolute free energy estimates (ΔG) using a maximum-likelihood estimator (MLE).

We can generate these results using the `generate_absolute_values()` method and then access all our absolute values using `get_absolute_dataframe()`.

In [ ]:
# First let's generate the absolute values for our calculated results
fe_map.generate_absolute_values()

In [ ]:
# Next we get the dataframe
dataframe = fe_map.get_absolute_dataframe()

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


The maximum-likelihood estimator requires that you have relative free energy data which weakly connects all the ligands. That is to say that a path from any ligand to any other ligand must be possible, even if it is through one or more intermediary ligands.

For more details on the maximum-likelihood estimator please see [H. Xu, *J. Chem. Inf. Model.*, 2019](https://pubs.acs.org/doi/abs/10.1021/acs.jcim.9b00528) and [I.M. Kenney and O. Beckstein, *Biophys. Reports*, 2023](https://doi.org/10.1016/j.bpr.2023.100120).

</div>

In [ ]:
# Let's look at the MLE generated values, these have been given the source "MLE"
mle_values = dataframe[dataframe['source'] == 'MLE']
mle_values.head()

In [ ]:
# The absolute dataframe also contains the experimental values we passed in
experimental_values = dataframe[dataframe['source'] == 'experimental']
experimental_values.head()

<div style="background-color:#fdecea; border-left: 6px solid #f44336; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

⚠️ **Note**


As you can see the MLE values show a ΔG of 1.45 kcal/mol whilst the experimental ΔG is -4.12 kcal/mol.

This is because the MLE-generated absolute values are always **centered around 0**, which means that they are offset from the experimental values. To compare the two sets of values, you would need to apply an experimental shift to the predicted values to align the means between the predicted and measured affinities. This is usually done by adding the mean of your experimental values to MLE-generated values.

</div>

Here is how you can go about shifting the MLE values by the mean of the experimental results:

In [ ]:
# We get the DG values for the experimental values
exp_DG = experimental_values["DG (kcal/mol)"]

# Our shift is the mean of the values
shift = exp_DG.mean()

# We then shift the MLE values in the dataframe
mle_values["DG (kcal/mol)"] += shift

# finally let's look at the first few ligands so we can compare against experiment
mle_values.head()

As you can see, after the shift ligand 0 is now ~ 0.22 kcal/mol from experiment.

Just like with ΔΔG results, Cinnabar offers an easy utility method to plot your ΔG results, 

In [ ]:
plotting.plot_DGs(fe_map, source="MLE", figsize=5, title="Absolute Free Energies")

#### Visualising error distributions with ECDF plots

Looking at scatter plots and aggregate error statistics such as RMSE or MUE provide a useful summary of prediction quality, but they can hide important information about the **distribution** of errors. An **Empirical Cumulative Distribution Function (ECDF)** plot addresses this by showing what fraction of predictions fall within a given error threshold.

For example, an ECDF plot lets you quickly answer questions like:
- What percentage of my ΔΔG predictions have an absolute error below 1 kcal/mol?
- How does the error distribution differ between my results and another set of results.

To calculate the ECDF of our DDGs, we can use the utility method `ecdf_plot_DDGs`:

In [ ]:
fig = plotting.ecdf_plot_DDGs(fe_map, nbootstraps=10000, ci=0.95, title="ECDF of ΔΔG absolute errors", figsize=5)

This yields an ECDF plot of "edgewise" (i.e. for each calculated edge) ΔΔG absolute error ECDF. The **shaded region** is an estimate of the confidence interval (set by the `ci` keyword, default of 95%), which is generated via a bootstrapping procedure (controlled by the `nbootstraps` keyword, default of 1000).

Similarly to the scatter plots, we can also get ECDF of our ΔG errors, also known as "nodewise ΔG absolute errors". This can be done using the `ecdf_plot_DGs` method:

In [ ]:
fig = plotting.ecdf_plot_DGs(fe_map, nbootstraps=10000, ci=0.95, title="ECDF of ΔG absolute errors", figsize=5)

#### Exploring cycle closure errors

Cycle closure analysis is a useful internal consistency check for relative binding free energy networks. For a closed loop of transformations, the signed sum of the calculated ΔΔG values should be zero (assuming perfect sampling). A large non-zero value indicates that at least one edge in the cycle is inconsistent with the others and may have convergence or sampling issues.

However, a low cycle closure error can also arise by chance from cancellation of errors, so it does not prove that every edge is well sampled. Cycle closure analysis should therefore be included as part of a larger post-simulation analysis pipeline.

<div style="background-color:#fdecea; border-left: 6px solid #f44336; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

⚠️ **Note**

Not all alchemical networks incorporate cycles! For example the minimum spanning network we used in the CLI example does not guarantee any cycles.

</div>

First let's look at the cycle-level closure errors, i.e. the error on each cycle. We can do this using the `get_cycle_closure_dataframe` method which reports one row per detected cycle. By default, cycles up to length 5 are considered; this can be changed using the argument `max_cycle_length`.

The column on each row are:

| Column Label | Description |
|:---|:---|
| `source` | The computational source for the edge, e.g. "openfe" |
| `cycle` | The labels of the ligands involved in the cycle |
| `cc (kcal/mol)` | The raw unsigned cycle closure error, calculated as the signed sum of the ΔΔG values around the cycle |
| `cc_per_edge (kcal/mol)` | `cc (kcal/mol)` normalised by `sqrt(number_of_edges_in_cycle)`. This makes it eassier to compare cycles with different lengths |
| `cc_unc_normalized` | `cc (kcal/mol)` normalized by the propagated uncertainty. Values much larger than 1 indicate closure errors that are large relative to the reported edge uncertainties in the cycle. This is similarr to `cc_per_edge (kcal/mol)` but does not assume the edges have comparable uncertainties |

In [ ]:
cycle_closure_df = fe_map.get_cycle_closure_dataframe()
cycle_closure_df.head(10)

We can plot a histogram of the `cc_per_edge (kcal/mol)` using the `plot_cycle_closure` method:

In [ ]:
fig = plotting.plot_cycle_closure(fe_map, filename=None, bin_width=0.25)

As we can see, the majority of the cycles have a cycle closure below 0.25 kcal/mol, which means that our results are likely well converged.

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


This tutorial only offers a brief look at all the capabilities that [Cinnabar](https://github.com/OpenFreeEnergy/cinnabar) offers.
To learn more about what you can do with this tool, have a look at the [documentation](https://cinnabar.openfree.energy/en/latest/index.html), especially the [tutorials](https://cinnabar.openfree.energy/en/latest/tutorials/).

</div>

<div style="background-color:#e8f5e9; border-left: 6px solid #4caf50; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

✏️ **Exercise**


Try using Cinnabar to analyze the data from the CLI results. For your convenience, the ΔΔG are stored in the file `cli_ddG_results.tsv`.

**Note:** You won't be able to look at cycle closure errors because the MST network does not have redundancies!


</div>

### Step 10: Exploring our simulation trajectories

As a final step in this tutorial, we can also look at the simuluation trajectories we generated when we ran our transformations.

As [mentioned previously](#Step-3:-Executing-simulations), our alchemical simulations create various artifacts, including a PDB file (named `hybrid_system.pdb` under the "SetupUnit" folder) describing the simulated system as well as a NetCDF file (named `simulation.nc` under the "SimulationUnit" folder) containing the trajectory data for our simulation.

Here we show how we can load this data into [MDAnalysis](https://github.com/MDAnalysis/mdanalysis) for further analysis using the [openfe-analysis](https://openfe-analysis.openfree.energy/en/latest/) package.

We first start by defining the paths to our Topology (PDB) and Trajectory (NetCDF) files. In this demo, we only provide inputs for one transformation (repeat 0 of `ligand_12` to `ligand_13`) which can be found under `api_outputs/results`:

In [ ]:
# First let's define the path to our topology (PDB) and trajectory (NetCDF) files

topology = (
    "api_output/results/repeat0/rbfe_ligand_12_complex_ligand_13_complex/"
    "shared_HybridTopologySetupUnit-53baacd7ab3f4ae8bbcc57bcb7a04ef1_attempt_0/"
    "hybrid_system.pdb"
)

trajectory = (
    "api_output/results/repeat0/rbfe_ligand_12_complex_ligand_13_complex/"
    "shared_HybridTopologyMultiStateSimulationUnit-d7036bb1b3554ca9a0b441d578f1759a_attempt_0/"
    "simulation.nc"
)

We can then read these into `MDAnalysis` using the `FEReader` class from `openfe-analysis`. The `FEReader` is a custom `MDAnalysis` coordinate reader specifically geared towards reading our simulation NetCDF files.

What is particularly unique with these NetCDF files is that they contain trajectory information for every discrete lambda state along the transformation. In our simulations, this means that there are **11 trajectories** each representing a lambdas 0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9 and 1.0.

Here we pass the `index` keyword to define which lambda (indexed from 0 to 10), we want to read. We also use `index_method="state"` to indicate that we are reading by lambda state.

In [ ]:
# Now we create our MDAnalysis universe for the first state (i.e. lambda=0) and last state (i.e. lambda=1)
import MDAnalysis as mda
from openfe_analysis import FEReader

# load in the lambda == 0 state
universe_lambda0 = mda.Universe(topology, trajectory, format=FEReader, index=0, index_method="state")

# we could also load in the last state (lambda == 1), in the following manner:
universe_lambda1 = mda.Universe(topology, trajectory, format=FEReader, index=10, index_method="state")

This loads all the trajectory data into MDAnalysis.

We can even visualise the trajectory using `nglview`:

In [ ]:
import nglview as nv

view = nv.show_mdanalysis(universe_lambda0)
view

As you can see by iterating through the trajectory, the protein and ligand move along the periodic box. This happens because the molecules have a certain amount of translation motion during the molecular dynamic simulation.

Whilst expected, this can be unwanted - particularly when we want to do analyses where we need our coordinates to be aligned, such as RMSF and RMSD calculations.

To remove translational motion and align our trajectory, we can apply [MDAnalysis Transformations](https://userguide.mdanalysis.org/stable/trajectories/transformations.html). In `openfe-analysis`, we have some built-in transformations which can be applied using `apply_complex_alignment_transformations` and `apply_ligand_alignment_transformations`. Briefly these:
1. **Unwrap:** Makes each molecule whole by repairing splits across periodic boundaries.
2. **Image:** Shifts molecules into the same periodic image.
3. **Align (protein-only):** Aligns all frames to a common reference (the first frame) by minimizing the protein c-alpha RMSD, removing overall translation and rotational motion so that any observed motion reflects internal conformational change only.

Here is how you would apply these to our newly loaded MDAnalysis Universe (**note:** we have loaded a "complex" transformation which has a protein, so we use `apply_complex_alignment_transformations` instead of `apply_ligand_alignment_transformations`):

In [ ]:
from openfe_analysis.utils.apply_transformations import apply_complex_alignment_transformations

# MDAnalysis has a built-in selection for protein residues
protein = universe_lambda0.select_atoms("protein")
# Our ligand is has UNK for a residue name, so we select by resname
ligand = universe_lambda0.select_atoms("resname UNK")

apply_complex_alignment_transformations(universe_lambda0, protein=protein, ligands=[ligand])

We can now visualise our simulation again and see that translational motions have been removed:

In [ ]:
view = nv.show_mdanalysis(universe_lambda0)
view

We can now directly use [MDAnalysis analysis methods](https://userguide.mdanalysis.org/stable/examples/analysis/README.html) directly to analyse our trajectory. For example, we can get the root mean square fluctuation of our protein backbone in the following manner:

In [ ]:
import matplotlib.pyplot as plt
from MDAnalysis.analysis import rms

# Note: RMSF only makes sense once translation/rotation are removed
# we did this already via the transformations we applied earlier
ca = universe_lambda0.select_atoms("protein and name CA and not resname ACE NME")

R = rms.RMSF(ca).run(verbose=True)

plt.plot(ca.resids, R.results.rmsf)
plt.xlabel("Residue")
plt.ylabel("RMSF (Å)")
plt.title("Per-residue RMSF, protein Cα atoms")

We can also get the center of mass distance from the ligand to the protein binding site like this:

In [ ]:
from MDAnalysis.lib.distances import calc_bonds
from MDAnalysis.analysis.base import AnalysisFromFunction

# We define a method that calculates the center of mass distance
def com_distance_fn(ligand, site):
    return calc_bonds(ligand.center_of_mass().reshape(1, 3),
                      site.center_of_mass().reshape(1, 3), box=ligand.universe.dimensions)[0]

# selections for the ligand and the protein binding site
ligand = universe_lambda0.select_atoms("resname UNK")
# we define the binding site as the protein residues within 8 angstrom of the ligand
binding_site = universe_lambda0.select_atoms("protein and byres around 8 resname UNK")

aff = AnalysisFromFunction(com_distance_fn, universe_lambda0.trajectory, ligand, binding_site)
aff.run(verbose=True)

plt.plot(aff.results.timeseries)
plt.xlabel("Frame")
plt.ylabel("COM distance (Å)")
plt.title("Ligand ↔ binding-site COM distance")

The center of mass distance gives us a proxy for how much the ligand moves in the binding site. As we can see the ligand is particularly mobile towards the end of the trajectory.

`openfe-analysis` also offers some built-in tools for analysis and plotting. Here we show how we can use one of these tools to get the 2D RMSD of our protein structure. A 2D RMSD is the pairwise RMSD between all analyzed frame pairs. A well-behaved simulation shows a uniform low-RMSD matrix. High RMSD values between certain frame pairs (visible as brighter patches in the plot) suggests the protein visited structurally distinct conformations during the simulation.

In [ ]:
# openfe-analysis has a built-in Protein2DRMSD analysis
from openfe_analysis.rmsd import Protein2DRMSD

protein = universe_lambda0.select_atoms("protein")
protein_2D_rmsd = Protein2DRMSD(protein).run()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# the Protein2DRMSD methods stores the results as a flat array
# so we unpack the 2D RMSD data into a 2D array
N = int((1 + np.sqrt(8 * len(protein_2D_rmsd.results.rmsd2d) + 1)) / 2)
arr = np.zeros((N, N))
arr[np.triu_indices_from(arr, k=1)] = protein_2D_rmsd.results.rmsd2d
arr += arr.T

# Now we can plot our results
plt.imshow(arr, cmap='viridis')
plt.xlabel('Frame')
plt.ylabel('Frame')
plt.colorbar(label=r'RMSD ($\AA$)')

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Info**


You can learn more about how to analyze your simulation trajectories by looking at the [openfe-analysis structural analysis tutorials](https://openfe-analysis.openfree.energy/en/latest/tutorials/structural_analysis.html).

</div>

<div style="background-color:#e8f5e9; border-left: 6px solid #4caf50; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

✏️ **Exercise**


Try running some of these analyses for other lambda states!


</div>

## Review & Conclusions

In this notebook we've planned two RBFE campaigns for the same MCL-1 fragment series:

- With the **OpenFE CLI**, in four commands (`charge-molecules`, `plan-rbfe-network`, `quickrun`, and `gather`),
  using our best-practice defaults throughout.
- With the **OpenFE Python API**, customising the atom mapper, network topology, partial charge
  method, force field, and simulation length.

Applying these tools to your own systems can be as easy as swapping out the `ligands.sdf` and `protein.pdb` files! Try it out!

<div style="background-color:#d9edf7; border-left: 6px solid #2196F3; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">

💡 **Next steps**


If you want to learn more about how to run other types of simulations with OpenFE, have a look at our [**SepTop**](https://docs.openfree.energy/en/stable/tutorials/septop_tutorial.html) and [**Absolute Binding Free Energy**](https://docs.openfree.energy/en/stable/tutorials/abfe_tutorial.html) tutorials!

</div>